# 10 — Volatility overlay

One volatility overlay (VMP) on the 8 core runs of notebook 08 and on 60/40. The registered tests compare each overlay with its base (H1) and with 60/40 (H2); performance, episodes and sensitivities are descriptive.

Setup. Run labels as in notebook 08: the covariance estimator is in brackets, and `BL(0.1,S)` is notebook 03's BL(0.1). `VMP(m)` is the overlay on run m.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb08_helpers as h8
import _nb10_helpers as h
from maplab import Panel, apply_style

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)
simple_returns = ml.to_simple_returns(prices)
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}  first live overlay date: {h.LIVE.date()}")

In [ ]:
reg = ml.inference.load_registration("nb10")
adj = [x["id"] for x in reg["hypothesis"] if x["kind"] == "adjusted_null"]
assert adj == h.H1_IDS + h.H2_IDS
ml.inference.render_registration(reg)

## §2 The overlay and the statistics

General form (Moreira and Muir 2017): scale a portfolio's return by the inverse of its recent variance, $r^{VM}_{t} = (k/\hat\sigma^2_{t-1})\,r_t$, with $k$ set so the managed and unmanaged series have the same unconditional volatility. That needs leverage and a full-sample constant.

Our version, registered in §1, uses neither. For run $m$ with daily net return $r_t$ and BIL return $f_t$:

$$\hat\sigma_t = \sqrt{252}\;\mathrm{sd}\big(r_s - f_s,\ s = t-20,\dots,t\big), \qquad \sigma^*_t = \mathrm{mean}\big(\hat\sigma_s,\ s \le t-1\big),$$

$$c_t = \min\big(1,\ \sigma^*_t / \hat\sigma_{t-1}\big), \qquad c_t = 1 \text{ for the first 252 days},$$

$$r^{VMP}_t = c_t\,r_t + (1-c_t)\,f_t - \kappa\,\lvert c_t - d_{t-1}\rvert, \qquad d_t = \frac{c_t\,(1+r_t)}{1 + c_t r_t + (1-c_t) f_t}, \qquad \kappa = 10\ \text{bp}.$$

$d_{t-1}$ is yesterday's risky fraction after drift, so the cost is charged on the exact trade. With $c \le 1$ the overlay only brakes: it moves part of the book into T-bills when trailing volatility is above its own history and never adds risk when it is below. The base method's trading costs are already in $r_t$; the overlay adds only the risky-to-BIL trade.

Tests: the delta-method Newey–West Sharpe difference, stationary bootstrap and Romano–Wolf step-down of notebook 08 §2, applied to $\Delta SR = SR(\mathrm{VMP}(m)) - SR(m)$ (H1) and $SR(\mathrm{VMP}(m)) - SR(60/40)$ (H2) on daily excess returns from 2010-02-02. Romano–Wolf uses $t^* = (\Delta SR^* - \Delta\widehat{SR})/\mathrm{se}_{NW}$, centred on the bootstrap's own estimator (ddof 1).

## §3 Runs and gates

The 8 core runs and 60/40, rebuilt with notebook 08's constructors. The gate checks their Sharpe ratios against notebook 08 at 2 dp in all three of notebook 08's windows.

In [ ]:
runs = h.build_base_runs(simple_returns, panel)
st = h8.summary_table(runs, h.CORE, split_ts, rf_daily)
ml.inference.check_reproduction(st, {k: h8.REPRO_KEYS[k] for k in h.CORE})
bst = h8.summary_table(runs, h.BENCH, split_ts, rf_daily)
ml.inference.check_reproduction(bst, h8.BENCH_REPRO_KEYS)
print(f"REPRODUCTION GATE: PASSED -- Sharpe to 2 dp for the {len(h.CORE)} core runs and 60/40, three windows.")

Overlays with the registered parameters. Identity gate: with $c \equiv 1$ the overlay returns the base series bit-for-bit; the net return rebuilt from $h$, $r$, BIL and the cost matches the overlay's; every overlay goes live on 2010-02-02.

In [ ]:
ov = h.build_overlays(runs, rf_daily)
all_runs = {**runs, **ov}
assert list(ov) == h.VMPS

idt = h.identity_table(runs, ov, rf_daily)
print(idt.to_string(float_format=F4))
assert idt["c1_identical"].all() and idt["warmup_c_all_one"].all() and idt["index_equal"].all()
assert float(idt["rebuild_max_abs"].max()) <= 1e-15
assert (idt["live_date"] == h.LIVE).all()
assert (idt["h_min"] > 0).all() and (idt["h_max"] <= 1.0).all()
print("IDENTITY GATE: PASSED")

No look-ahead: a +5% shock to the base return on 2020-03-16 leaves $c$ unchanged through that day and moves it the next day.

In [ ]:
for name in ("EW", "60/40"):
    sc = h.shock_check(runs, rf_daily, name, "2020-03-16")
    print(f"{name:<6} max |Δc| through the shock day = {sc['max_abs_dc_through_at']}   Δc next day = {sc['dc_next_day']:+.4f}")
    assert sc["max_abs_dc_through_at"] == 0.0 and sc["dc_next_day"] != 0.0
print("LOOK-AHEAD GATE: PASSED")

Scoring windows from the first live date, with the Newey–West lags used in §5 and §6.

In [ ]:
XW = h.scoring_frames(all_runs, h.BASES + h.VMPS, rf_daily, split_ts)
for w, Xw in XW.items():
    print(f"{w:<18} T={len(Xw):>5}  first={Xw.index[0].date()}  last={Xw.index[-1].date()}  NW lag={ml.robust.nw_lag(len(Xw))}")
assert [len(Xw) for Xw in XW.values()] == [4086, 3252, 834]
assert [ml.robust.nw_lag(len(Xw)) for Xw in XW.values()] == [9, 8, 6]
assert XW["full"].index[0] == h.LIVE

### 3.1 Exposure

Target fraction $c$, held fraction $h$ (equal under daily updates), share of days braking ($h < 1$), the lowest $c$ and its date, and the overlay's own turnover and cost drag, full window from 2010-02-02.

In [ ]:
ex = h.exposure_table(ov, split_ts)
exf = ex.xs("full", level="window")
print(exf.drop(columns="mean_h").to_string(float_format=F4))
print()
print(ex[["mean_c", "share_braking"]].unstack("window").reindex(h.VMPS).to_string(float_format=F4))

for lab, o in ov.items():
    assert np.array_equal(o["h"].to_numpy(), o["c"].to_numpy())
    assert abs(o["diag"]["share_braking"] - exf.loc[lab, "share_braking"]) <= 1e-15
assert (exf["mean_c"] <= 1.0).all()
assert exf["min_c_date"].between(pd.Timestamp("2020-03-25"), pd.Timestamp("2020-04-07")).all()
print("EXPOSURE CHECKS: PASSED")

Mean $c$ is 0.87–0.94, lowest for MaxSharpe(S): the nine overlays brake on 28–42% of days and is fully invested otherwise. Every lowest $c$ falls between 2020-03-25 and 2020-04-07, at 0.14–0.21. Cost drag is 19–28 bp/yr, highest for MaxSharpe(S); the 18–27 bp/yr in §1 comes from the pre-check, whose averaging span is not recorded.

F1: held fraction $h$ over time, as the median and range across the 8 core overlays, with VMP(60/40). The table below it gives the mean $h$ per calendar year.

In [ ]:
fig = h.exposure_figure(ov, h.CORE)
plt.show()
yr = h.exposure_by_year(ov, h.CORE)
print(yr.to_string(float_format=F4))
assert len(fig.axes) == 1 and len(fig.axes[0].get_lines()) == 2

The median core overlay holds least in 2022 (0.77) and 2020 (0.85) among full calendar years; 2026 to date (January–April) is 0.84. Its lowest single day is 2020-04-07 (median $h$ 0.17).

## §4 Performance

Base runs, their overlays, 60/40 and VMP(60/40), from 2010-02-02 in the three windows. For a VMP row, turnover and cost drag are the overlay's own risky-to-BIL trading; the base method's trading is already in its net return.

In [ ]:
ORDER = [x for m in h.BASES for x in (m, h.vmp(m))]
perf = h.perf_table(all_runs, ORDER, split_ts, rf_daily)
perf = perf.reindex(pd.MultiIndex.from_product([ORDER, list(h.WINDOWS)], names=["strategy", "window"]))
PERF_COLS = ["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "ann_turnover", "cost_drag"]
print(perf[PERF_COLS].to_string(float_format=F4))

gap = 0.0
for w, Xw in XW.items():
    sr = Xw.mean() / Xw.std(ddof=1) * np.sqrt(ml.TRADING_DAYS)
    gap = max(gap, float((sr[ORDER] - perf.xs(w, level="window").loc[ORDER, "sharpe"]).abs().max()))
print(f"max |point SR (ddof 1) - table Sharpe| over all rows and windows = {E3(gap)}")
assert gap <= 1e-10 and not perf[PERF_COLS].isna().any().any()

In the full window the overlay lowers both volatility and return for all nine runs; volatility falls to 73% (MaxSharpe(S)) to 87% (MDP(S)) of the base. Sharpe rises for four runs (60/40 +0.09, BMV(0.3) +0.05, MaxSharpe(S) +0.02, HRP(S) +0.01) and falls for five, most for MDP(S) (−0.08) and GMV(S) (−0.07). Maximum drawdown is shallower for seven; GMV(S) and MDP(S) are slightly deeper. VMP(60/40) has the highest full-window Sharpe of the 18 rows (0.96), but in the test window it is below 60/40 (0.85 against 1.00).

F2: each arrow runs from a base run to its overlay in (volatility, return), full window from 2010-02-02.

In [ ]:
fig = h.arrows_figure(perf, h.BASES)
plt.show()
assert len(fig.axes) == 1

The three largest volatility cuts, MaxSharpe(S), 60/40 and BMV(0.3), are three of the four runs whose Sharpe rises; the two smallest, MDP(S) and GMV(S), are the two largest Sharpe losses.

## §5 H1: overlay vs base

Registered test (§1): $\Delta SR = SR(\mathrm{VMP}(m)) - SR(m)$ on the full window, delta-method Newey–West $t$, Romano–Wolf over the 8, with the test-window $\Delta SR$, the bootstrap 95% interval and Holm (descriptive).

In [ ]:
SR_star, IDX_full = h.bootstrap_full(XW, h.BASES + h.VMPS)
assert sum(c.shape[0] for c in IDX_full) == h.B and all(c.shape[1] == len(XW["full"]) for c in IDX_full)
print(f"stationary bootstrap: B={h.B}, mean block {h.BLOCK}, seed {h.SEED}, {SR_star.shape[1]} series, one shared index draw on the full window")

In [ ]:
DSR_COLS = ["dSR_full", "se", "t", "p_raw", "p_holm", "p_rw", "dSR_test", "same_sign_test", "ci_lo", "ci_hi", "boot_p"]
h1, h1_stats, t1 = h.dsr_family(XW, h.H1_PAIRS, h.H1_IDS, SR_star)
assert list(h1.index) == h.H1_IDS and (h1["L"] == 9).all() and (h1["L_test"] == 6).all()
print(h1[["b"] + DSR_COLS].to_string(float_format=F4))
print(f"ddof gap: max |dSR (delta, ddof 0) - dSR (bootstrap centre, ddof 1)| = {E3(float((h1['dSR_full'] - h1['d_hat_ddof1']).abs().max()))}")

ΔSR runs from −0.08 (MDP(S)) to +0.05 (BMV(0.3)); the largest $|t|$ is 1.23 and the smallest Romano–Wolf p 0.451. The standard errors, 0.07 (MDP(S)) to 0.12 (ERC(S)), are below notebook 08's 0.10 to 0.26 because VMP(m) equals m on most days; an unadjusted 5% rejection would still need a gap of about 0.13 to 0.24, against an observed maximum of 0.08. The test-window ΔSR keeps the full-window sign for 7 of the 8 (not BMV(0.3)).

## §6 H2: overlay vs 60/40

Same test for $\Delta SR = SR(\mathrm{VMP}(m)) - SR(60/40)$. VMP(60/40) − 60/40 is shown below the family as a descriptive row; it is not tested.

In [ ]:
h2, h2_stats, t2 = h.dsr_family(XW, h.H2_PAIRS, h.H2_IDS, SR_star)
assert list(h2.index) == h.H2_IDS and (h2["L"] == 9).all() and (h2["L_test"] == 6).all()
print(h2[["a"] + DSR_COLS].to_string(float_format=F4))
print(f"ddof gap: max |dSR (delta, ddof 0) - dSR (bootstrap centre, ddof 1)| = {E3(float((h2['dSR_full'] - h2['d_hat_ddof1']).abs().max()))}")

d6040, _, _ = h.dsr_family(XW, [h.D_PAIR], ["D_6040"], SR_star)
print("\ndescriptive, not tested:")
print(d6040[["a", "b", "dSR_full", "se", "t", "p_raw", "dSR_test", "ci_lo", "ci_hi"]].to_string(float_format=F4))

Every VMP(m) has a lower full-window Sharpe than 60/40, from −0.35 (BL(0.1,S)) to −0.003 (BMV(0.3)); the largest $|t|$ is 1.93 (BL(0.1,S), Romano–Wolf p 0.198). BL(0.1,S)'s bootstrap 95% interval just excludes 0 (upper end −0.008) while $|t| < 2$: its upper bootstrap tail (0.34) is shorter than 1.96 × se (0.35). In the test window VMP(MaxSharpe(S)) and VMP(BMV(0.3)) are above 60/40. The standard errors, 0.13 to 0.28 against H1's 0.07 to 0.12, leave little power, so these nulls are weak evidence of equal Sharpe ratios. VMP(60/40) − 60/40 (descriptive) is +0.09 (t 0.87) in the full window and −0.14 in the test window.

## Verdict table

Verdicts applied in code from the rules registered in §1.

In [ ]:
verdict = ml.inference.evaluate(reg, {**h1_stats, **h2_stats})
assert len(verdict) == 16
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    print(verdict.to_string(index=False))
print()
print(verdict["verdict"].value_counts().to_string())

All 16 hypotheses: NULL HOLDS. No overlay changes its base run's Sharpe ratio detectably, and no overlaid core run's Sharpe ratio is detectably different from 60/40's; as in notebook 08, NULL HOLDS is not evidence of equal Sharpe ratios.

F4: $\Delta SR$ with bootstrap 95% intervals, H1 and H2 side by side, by base method.

In [ ]:
fp = pd.concat([
    h1.assign(family="H1: VMP(m) − m", label=list(h1["b"])),
    h2.assign(family="H2: VMP(m) − 60/40", label=[a[4:-1] for a in h2["a"]]),
])
fig = ml.plotting.forest_plot(fp, "label", "d_hat_ddof1", "ci_lo", "ci_hi", group="family", ref=0.0,
                              title="F4 — ΔSR with bootstrap 95% intervals, full window from 2010-02-02", xlabel="ΔSR")
plt.show()

The H1 intervals straddle 0 for all eight. The H2 intervals are wider for every method and centred below 0; only BL(0.1,S)'s excludes 0.

## §7 Episodes

The two registered episodes, February–April 2020 and calendar 2022: cumulative return and in-window maximum drawdown of each base and its overlay, with the overlay's mean and minimum held fraction.

In [ ]:
ep = h.episode_table(all_runs, ORDER)
assert ep.shape[0] == 2 * len(ORDER)
rows = []
for e in h.EPISODES:
    for m in h.BASES:
        b, v = ep.loc[(m, e)], ep.loc[(h.vmp(m), e)]
        rows.append({"episode": e, "run": m, "ret_base": b["cum_return"], "ret_vmp": v["cum_return"],
                     "dd_base": b["max_dd"], "dd_vmp": v["max_dd"], "mean_h": v["mean_h"], "min_h": v["min_h"]})
epc = pd.DataFrame(rows).set_index(["episode", "run"])
print(epc.to_string(float_format=F4))
print()
print(ep.xs("EW", level="run")[["first", "last"]].to_string())

In February–April 2020 the overlay cuts the drawdown for all nine runs, to 42% (ERC(S)) to 57% (MDP(S)) of the base, yet cumulative return over the episode is lower for 4 of the nine (GMV(S), MaxSharpe(S), MDP(S) and 60/40). In 2022 the drawdown is shallower for all nine, most for 60/40 (−21.0% to −13.5%), and mean $h$ is 0.58 (60/40) to 0.87 (MDP(S)).

F3: drawdown within each episode (median across the 8 core runs, base solid and VMP dashed; 60/40 and VMP(60/40) in the 60/40 colour) and held fraction $h$.

In [ ]:
fig = h.episode_figure(all_runs, h.CORE)
plt.show()
assert len(fig.axes) == 4

In 2020 the median core drawdown troughs on 2020-03-19 and the median core $h$ bottoms on 2020-04-07, 13 trading days later, so the overlay is most underweight during the rebound. In 2022 the median core $h$ is below 1 on 89% of trading days.

## §8 Sensitivities

The four registered variants on all nine runs, from 2010-02-02; never tested. Cap 1.5 borrows at BIL with no spread, so it is optimistic.

In [ ]:
sens = h.sensitivity_table(runs, rf_daily, split_ts, ov)
VARS = ["registered"] + list(h.SENS)
for w in ("full", "test (> split)"):
    dv = sens.xs(w, level="window")["dSR_vs_base"].unstack("variant").loc[h.BASES, VARS]
    print(f"ΔSR vs base, {w}:")
    print(dv.to_string(float_format=F4))
    print()
sf = sens.xs("full", level="window")
print("mean h, full:")
print(sf["mean_h"].unstack("variant").loc[h.BASES, VARS].to_string(float_format=F4))
print()
print("overlay cost drag, bp/yr, full:")
print(sf["cost_drag_bps"].unstack("variant").loc[h.BASES, VARS].to_string(float_format=lambda v: f"{v:.1f}"))

P_full = perf.xs("full", level="window")
reg_gap = max(abs(float(sf.loc[("registered", n), "dSR_vs_base"])
                  - float(P_full.loc[h.vmp(n), "sharpe"] - P_full.loc[n, "sharpe"])) for n in h.BASES)
print(f"\nregistered row vs §4 table: max |ΔSR difference| = {E3(reg_gap)}")
assert reg_gap <= 1e-10
assert (sf["dSR_vs_registered"].xs("registered", level="variant") == 0.0).all()

The slower variants trade far less: overlay cost drag is 6–10 bp/yr with a 63-day window and 7–14 with month-end updates, against 19–28 registered. Inverse variance brakes harder (mean $h$ 0.79–0.90); cap 1.5 holds 1.08–1.20 on average at 64–77 bp/yr. Across all 45 variant-run cells the full-window ΔSR vs base stays between −0.15 (MDP(S), inverse variance) and +0.09 (60/40, inverse variance); in the test window it is negative under every variant for 5 runs (EW, BMV(0.3), BL(0.1,S), ERC(S) and 60/40).

F5: full-window Sharpe of each variant minus the registered overlay's, per run.

In [ ]:
fig = h.sensitivity_heatmap(sens)
plt.show()
assert len(fig.axes) == 2

Against the registered overlay, Sharpe is higher for 6 of 9 runs with the 63-day window and 5 of 9 with month-end updates, most for MDP(S) and GMV(S) in both; it is lower for 8 of 9 with inverse variance and 6 of 9 with cap 1.5.

## §9 TAKEAWAY

- The overlay is built on each run's net return, reproduces it exactly at $c = 1$ and uses no future data; it brakes on 28–42% of days at 19–28 bp/yr.
- From 2010-02-02 it lowers volatility (to 73–87% of the base) and return for all nine runs; Sharpe moves by −0.08 (MDP(S)) to +0.09 (60/40).
- H1 and H2: 16 of 16 NULL HOLDS. The overlay does not detectably change any run's Sharpe ratio, and no overlaid core run is detectably different from 60/40; H2 has little power.
- Drawdowns are shallower for all nine in both registered episodes (42–57% of the base in 2020), but in 2020 the overlay is most underweight after the trough.
- Slower variants cut the cost drag to 6–14 bp/yr and narrow the MDP(S) and GMV(S) losses; no variant moves full-window ΔSR vs base outside −0.15 to +0.09.

On these methods the volatility overlay is drawdown control, not a Sharpe improvement: it buys shallower crisis drawdowns with lower return, and its Sharpe effect is too small to detect.